In [1]:
import json
import pandas as pd
from datetime import datetime

The exported .json file from Wireshark a lot of times contains duplicate keys in an object, e.g. when there are multiple quic packets in one UDP packet.

In the .json we can see something like:

    "quic": {
    ...
    "quic.frame": { "quic.frame_type": "0x02", ... },
    "quic.frame": { "quic.frame_type": "0x18", ... },
    "quic.frame": { "quic.frame_type": "0x1e", ... },
    ...
    }

For this reason we need a custom JSON parser hook, that handles this problem, as the standard json.load() just overwrites the previous object in this case

In [2]:
def as_list_on_duplicate_keys(ordered_pairs):
    """
    A custom JSON object_pairs_hook that collects values for duplicate
    keys into a list.
    """
    d = {}
    for k, v in ordered_pairs:
        if k in d:
            if isinstance(d[k], list):
                d[k].append(v)
            else:
                d[k] = [d[k], v]
        else:
            d[k] = v
    return d


In [3]:
import os
def analyze_quic_capture(json_file_path, encoding='utf-8'):
    """
    Analyzes a decrypted QUIC packet capture from a Wireshark JSON export
    and extracts a set of high-level features for connection migration analysis.

    Args:
        json_file_path (str): The path to the JSON file.

    Returns:
        dict: A dictionary containing the extracted features for the flow.
              Returns None if the capture is empty or invalid.
    """
    with open(json_file_path, 'r', encoding=encoding) as f:
        packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)

    if not packets:
        print("Capture file is empty.")
        return None
    
    base_name = os.path.basename(json_file_path)
    file_id = os.path.splitext(base_name)[0]

    try:
        capture_number = int(file_id.split('_')[-1])
    except (ValueError, IndexError):
        capture_number = None

    features = {}
    features['file_id'] = capture_number
    # --- 1. Initialization and Initial Packet Analysis ---
    first_packet = packets[0]['_source']['layers']
    last_packet = packets[-1]['_source']['layers']

    initial_ip_client = first_packet['ip']['ip.src']
    initial_ip_server = first_packet['ip']['ip.dst']
    initial_port_client = int(first_packet['udp']['udp.srcport'])
    initial_port_server = int(first_packet['udp']['udp.dstport'])
    
    time_first_epoch = float(first_packet['frame']['frame.time_epoch'])
    time_last_epoch = float(last_packet['frame']['frame.time_epoch'])

    features['initial_ip_client'] = initial_ip_client
    features['initial_ip_server'] = initial_ip_server
    features['initial_port_client'] = initial_port_client
    features['initial_port_server'] = initial_port_server
    features['time_first'] = datetime.fromtimestamp(time_first_epoch).isoformat(sep='T', timespec='microseconds').replace(":", "-")
    features['time_last'] = datetime.fromtimestamp(time_last_epoch).isoformat(sep='T', timespec='microseconds').replace(":", "-")
    features['connection_duration'] = (time_last_epoch - time_first_epoch) * 1000  # in ms

    # Initialize counters and flags
    bytes_sent_client = 0
    bytes_sent_server = 0
    packets_sent_client = 0
    packets_sent_server = 0
    quic_packets_sent_server = 0
    quic_packets_sent_client = 0


    # Stream-related counters: number of streams opened
    total_bidi_streams_client_init = 0
    total_bidi_streams_server_init = 0
    total_udi_streams_client_init = 0
    total_udi_streams_server_init = 0

    udi_streams = set()
    bidi_streams = set()
    bytes_bidi_streams_client_init_client_sent = 0
    bytes_bidi_streams_client_init_server_sent = 0
    bytes_bidi_streams_server_init_client_sent = 0
    bytes_bidi_streams_server_init_server_sent = 0
    bytes_udi_streams_client_init = 0
    bytes_udi_streams_server_init = 0

    padding_bytes_in_validation_pc = 0
    padding_bytes_in_validation_pr = 0
    mtu = 0

    # Specific QUIC packet counters
    ack_sent_client = 0
    ack_sent_server = 0
    crypto_sent_client = 0
    crypto_sent_server = 0
    handshake_done_client = 0
    handshake_done_server = 0
    path_challenge_sent_client = 0
    path_challenge_sent_server = 0
    path_response_sent_client = 0
    path_response_sent_server = 0

    features['version_negotiation_occurred'] = 0
    features['retry_occurred'] = 0
    features['new_connection_ids_issued_server'] = 0
    features['retired_cid_count_client'] = 0
    features['retired_cid_count_server'] = 0
    features['new_connection_ids_issued_server'] = 0
    features['new_connection_ids_issued_client'] = 0
    
    # State variables
    migrated = False
    migrated_ip_client = None
    migrated_port_client = None
    time_to_migration = None
    packets_before_migration = 0
    
    handshake_start_time = None
    handshake_end_time = None # We will find the timestamp of the LAST 'Finished' message
    
    path_challenge_data = None
    path_challenge_time = None
    migration_start_time = None
    last_pr_time = None
    
    app_data_before_migration = 0


    # --- 2. Iterate Through All Packets ---
    for i, pkt_data in enumerate(packets):
        layers = pkt_data['_source']['layers']
        
        # Basic packet info
        current_time = float(layers['frame']['frame.time_epoch'])
        packet_len = int(layers['frame']['frame.len'])
        
        src_ip = layers['ip']['ip.src']
        dst_ip = layers['ip']['ip.dst']

        src_port = int(layers['udp']['udp.srcport'])
        dst_port = int(layers['udp']['udp.dstport'])
        
        if not migrated and \
           (dst_ip == initial_ip_server and dst_port == initial_port_server) and \
           (src_ip != initial_ip_client or src_port != initial_port_client):
            
            migrated = True
            migration_start_time = current_time

            migrated_ip_client = src_ip
            migrated_port_client = src_port
            
            time_to_migration = (current_time - time_first_epoch) * 1000
            packets_before_migration = i
            
            ip_changed = src_ip != initial_ip_client
            port_changed = src_port != initial_port_client
            if ip_changed and port_changed:
                features['migration_type'] = 'IP_AND_PORT'
            elif ip_changed:
                features['migration_type'] = 'IP_ONLY'
            elif port_changed:
                features['migration_type'] = 'PORT_ONLY'


        is_client_pkt = (src_ip == initial_ip_client or src_ip == migrated_ip_client)

        # Update byte and packet counters
        if is_client_pkt:
            bytes_sent_client += packet_len
            packets_sent_client += 1
        else:
            bytes_sent_server += packet_len
            packets_sent_server += 1
        quic_packet_list = layers['quic']
        if not isinstance(quic_packet_list, list):
            quic_packet_list = [quic_packet_list]

        # 2. Loop through each QUIC packet within the UDP datagram.
        for quic_packet in quic_packet_list:
            if quic_packet.get('quic.version') == '0x00000000':
                features['version_negotiation_occurred'] = 1
            if quic_packet.get('quic.long.packet_type') == '3': # Retry packet
                features['retry_occurred'] = 1
            if quic_packet.get('quic.long.packet_type') == '0': # Initial packet
                if handshake_start_time is None:
                    handshake_start_time = current_time

            # Analyze QUIC frames if they exist
            # 3. Get frames from the current quic_packet, not from layers['quic'].
            quic_frames = quic_packet.get('quic.frame', [])
            if not isinstance(quic_frames, list): # If there's only one frame, it's a dict
                quic_frames = [quic_frames]
            
            # These flags help identify which packet contains padding
            is_path_challenge_pkt = any(f.get('quic.frame_type') == '0x000000000000001a' for f in quic_frames)
            is_path_response_pkt = any(f.get('quic.frame_type') == '0x000000000000001b' for f in quic_frames)
                
            for frame in quic_frames:
                frame_type = frame.get('quic.frame_type')
                if not frame_type: continue # Skip if frame is empty

                if is_client_pkt:
                    quic_packets_sent_client += 1
                else:
                    quic_packets_sent_server += 1
                
                if frame_type == '0x0000000000000006': # CRYPTO frame
                    if is_client_pkt:
                        crypto_sent_client += 1
                    else:
                        crypto_sent_server += 1
                
                if frame_type in ('0x0000000000000002', '0x0000000000000003'): # ACK frame
                    if is_client_pkt:
                        ack_sent_client += 1
                    else:
                        ack_sent_server += 1
                
                if frame_type == '0x000000000000001e': # HANDSHAKE_DONE frame
                    if handshake_end_time is None:
                        handshake_end_time = current_time
                    if is_client_pkt:
                        handshake_done_client += 1
                    else:
                        handshake_done_server += 1
                
                if frame_type == '0x0000000000000018': # NEW_CONNECTION_ID
                    if not is_client_pkt:
                        features['new_connection_ids_issued_server'] += 1
                    else:
                        # Note: clients do not issue CIDs, but keeping your logic
                        features['new_connection_ids_issued_client'] += 1
                
                if frame_type == '0x0000000000000019': # RETIRE_CONNECTION_ID
                    if is_client_pkt:
                        features['retired_cid_count_client'] += 1
                    else:
                        features['retired_cid_count_server'] += 1

                if frame_type == '0x000000000000001a': # PATH_CHALLENGE
                    if is_client_pkt:
                        features['path_validation_initiated'] = 1
                        path_challenge_data = frame['quic.path_challenge.data']
                        path_challenge_time = current_time
                        mtu = max(mtu, packet_len)
                    if is_client_pkt:
                        path_challenge_sent_client += 1
                    else:
                        path_challenge_sent_server += 1

                if frame_type == '0x000000000000001b': # PATH_RESPONSE
                    if not is_client_pkt:
                        if path_challenge_data and frame['quic.path_response.data'] == path_challenge_data:
                            features['first_path_validation_response_latency'] = (current_time - path_challenge_time) * 1000
                            #features['migration_duration'] = (current_time - migration_start_time) * 1000
                    last_pr_time = current_time
                    if is_client_pkt:
                        path_response_sent_client += 1
                    else:
                        path_response_sent_server += 1
            
                # Padding is its own frame type, not a field on other frames.
                if frame_type == '0x0000000000000000': # PADDING frame
                    padding_len = int(frame.get('quic.padding_length', 1)) # Padding can be 1 byte if length is omitted
                    if is_path_challenge_pkt:
                        padding_bytes_in_validation_pc = padding_len
                    if is_path_response_pkt:
                        padding_bytes_in_validation_pr = padding_len
                
                if frame_type in ('0x0000000000000008', '0x0000000000000009', '0x000000000000000a', '0x000000000000000b', '0x000000000000000c', '0x000000000000000d', '0x000000000000000e', '0x000000000000000f'): # STREAM frames
                    initiator = int(frame.get('quic.stream.stream_id_tree').get('quic.stream.initiator'),0)
                    direction = int(frame.get('quic.stream.stream_id_tree').get('quic.stream.direction'),0)
                    bytes = int(frame.get('quic.stream.length', 0))
                    id = int(frame.get('quic.stream.stream_id', 0))
                    if not migrated:
                        app_data_before_migration += bytes
                    if direction == 0:
                        if initiator == 0:
                            if id not in bidi_streams:
                                total_bidi_streams_client_init += 1
                                bidi_streams.add(id)
                            if is_client_pkt:
                                bytes_bidi_streams_client_init_client_sent += bytes
                            else:
                                bytes_bidi_streams_client_init_server_sent += bytes
                        else:
                            if id not in bidi_streams:
                                total_bidi_streams_server_init += 1
                                bidi_streams.add(id)
                            if is_client_pkt:
                                bytes_bidi_streams_server_init_client_sent += bytes
                            else:
                                bytes_bidi_streams_server_init_server_sent += bytes
                    else:
                        if initiator == 0:
                            if id not in udi_streams:
                                total_udi_streams_client_init += 1
                            bytes_udi_streams_client_init += bytes
                        else:
                            if id not in udi_streams:
                                total_udi_streams_server_init += 1
                            bytes_udi_streams_server_init += bytes

                if frame_type in ('0x000000000000001c', '0x000000000000001d'): # CONNECTION_CLOSE
                    features['connection_close_type'] = 'CLIENT_CLOSE' if is_client_pkt else 'SERVER_CLOSE'    
            


    # --- 3. Final Calculations and Assembly ---
    features['bytes_sent_client'] = bytes_sent_client
    features['bytes_sent_server'] = bytes_sent_server
    features['packets_sent_client'] = packets_sent_client
    features['packets_sent_server'] = packets_sent_server
    features['quic_packets_sent_client'] = quic_packets_sent_client
    features['quic_packets_sent_server'] = quic_packets_sent_server
    
    if handshake_end_time:
        features['handshake_duration'] = (handshake_end_time - handshake_start_time) * 1000
    else:
        features['handshake_duration'] = None # Handshake did not complete or was not found
        
    features['time_to_migration'] = time_to_migration
    features['migration_duration'] = (last_pr_time - migration_start_time) * 1000

    features['packets_before_migration'] = packets_before_migration
    features['app_data_bytes_before_migration'] = app_data_before_migration
    
    # Set migration-related features to 0 or null if no migration occurred
    if not migrated:
        features['migration_type'] = 'NO_CHANGE'
        features['time_to_migration'] = 0
        features['packets_before_migration'] = 0
        features['migration_duration'] = 0
        features['path_validation_initiated'] = 0
        features['first_' \
        'first_path_validation_response_latency'] = 0

    # These features couldn't be accurately determined from this specific JSON structure but are included as placeholders
    features['padding_bytes_in_validation_pc'] = padding_bytes_in_validation_pc
    features['padding_bytes_in_validation_pr'] = padding_bytes_in_validation_pr
    features['mtu'] = mtu
    
    features['total_bidi_streams_client_init'] = total_bidi_streams_client_init
    features['total_bidi_streams_server_init'] = total_bidi_streams_server_init
    features['total_udi_streams_client_init'] = total_udi_streams_client_init
    features['total_udi_streams_server_init'] = total_udi_streams_server_init
    features['bytes_bidi_streams_client_init_client_sent'] = bytes_bidi_streams_client_init_client_sent
    features['bytes_bidi_streams_client_init_server_sent'] = bytes_bidi_streams_client_init_server_sent
    features['bytes_bidi_streams_server_init_client_sent'] = bytes_bidi_streams_server_init_client_sent
    features['bytes_bidi_streams_server_init_server_sent'] = bytes_bidi_streams_server_init_server_sent
    features['bytes_udi_streams_client_init'] = bytes_udi_streams_client_init
    features['bytes_udi_streams_server_init'] = bytes_udi_streams_server_init

    features['ack_sent_client'] = ack_sent_client
    features['ack_sent_server'] = ack_sent_server
    features['crypto_sent_client'] = crypto_sent_client
    features['crypto_sent_server'] = crypto_sent_server
    features['handshake_done_client'] = handshake_done_client
    features['handshake_done_server'] = handshake_done_server
    features['path_challenge_sent_client'] = path_challenge_sent_client
    features['path_challenge_sent_server'] = path_challenge_sent_server
    features['path_response_sent_client'] = path_response_sent_client
    features['path_response_sent_server'] = path_response_sent_server
    features['app_data_bytes_before_migration'] = app_data_before_migration

    


    # Final check for connection close type
    if 'connection_close_type' not in features:
        features['connection_close_type'] = 'IDLE_TIMEOUT' # Default assumption
        
    return features

In [4]:
import os

def process_quic_capture(json_file_path):
    """
    Determines the correct encoding (UTF-8 or UTF-16) for a JSON file
    and then calls the analysis function.
    """

    if not os.path.exists(json_file_path) or os.path.getsize(json_file_path) == 0:
        print(f"Skipping '{json_file_path}': File is empty or does not exist.\n")
        return

    try:
        extracted_features = analyze_quic_capture(json_file_path, encoding='utf-8')
        return extracted_features

    except UnicodeDecodeError:
        try:
            extracted_features = analyze_quic_capture(json_file_path, encoding='utf-16')
            return extracted_features
        except (json.JSONDecodeError, UnicodeDecodeError) as e:
            print(f"    -> ERROR: Failed to process '{os.path.basename(json_file_path)}' as UTF-16. Error: {e}")

    except json.JSONDecodeError as e:

        print(f"    -> ERROR: File is UTF-8 but has invalid JSON. Error: {e}")

In [13]:
pd.set_option('display.max_columns', None)

In [6]:
json_file = 'captures_json\quiche\quiche_capture_20.json'
extracted_features = process_quic_capture(json_file)

In [7]:
if extracted_features:
    df = pd.DataFrame([extracted_features], index=[extracted_features['file_id']])
    print("Extracted Features (Corrected Migration & Direction Logic):")
    display(df)

Extracted Features (Corrected Migration & Direction Logic):


,file_id,initial_ip_client,initial_ip_server,initial_port_client,initial_port_server,time_first,time_last,connection_duration,version_negotiation_occurred,retry_occurred,new_connection_ids_issued_server,retired_cid_count_client,retired_cid_count_server,new_connection_ids_issued_client,migration_type,path_validation_initiated,first_path_validation_response_latency,connection_close_type,bytes_sent_client,bytes_sent_server,packets_sent_client,packets_sent_server,quic_packets_sent_client,quic_packets_sent_server,handshake_duration,time_to_migration,migration_duration,packets_before_migration,app_data_bytes_before_migration,padding_bytes_in_validation_pc,padding_bytes_in_validation_pr,mtu,total_bidi_streams_client_init,total_bidi_streams_server_init,total_udi_streams_client_init,total_udi_streams_server_init,bytes_bidi_streams_client_init_client_sent,bytes_bidi_streams_client_init_server_sent,bytes_bidi_streams_server_init_client_sent,bytes_bidi_streams_server_init_server_sent,bytes_udi_streams_client_init,bytes_udi_streams_server_init,ack_sent_client,ack_sent_server,crypto_sent_client,crypto_sent_server,handshake_done_client,handshake_done_server,path_challenge_sent_client,path_challenge_sent_server,path_response_sent_client,path_response_sent_server
20,20,127.0.0.2,127.0.0.1,54110,4433,2025-10-20T18-32-53.659294,2025-10-20T18-32-53.678830,19.536018,1,1,1,0,0,1,IP_AND_PORT,1,1.641989,CLIENT_CLOSE,7269,4649,14,12,19,19,11.227846,12.987137,2.469063,13,47,1294,1294,1382,1,0,4,4,72,308,0,0,47,47,5,5,3,4,0,1,1,1,1,1


#### Loading one json for test

In [8]:
json_file = 'captures_json\quiche\quiche_capture_20.json'
import os
if os.path.getsize(json_file) > 0:
    with open(json_file, 'r', encoding='utf-8') as f:
        try:
            extracted_features = analyze_quic_capture(json_file, encoding='utf-8')
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
else:
    print("The JSON file is empty.")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte

In [ ]:
if extracted_features:
    df = pd.DataFrame([extracted_features])#, index=[extracted_features['ID']])
    print("Extracted Features (Corrected Migration & Direction Logic):")
    display(df)

Extracted Features (Corrected Migration & Direction Logic):


,ID,initial_ip_client,initial_ip_server,initial_port_client,initial_port_server,time_first,time_last,connection_duration,version_negotiation_occurred,retry_occurred,new_connection_ids_issued_server,retired_cid_count_client,retired_cid_count_server,new_connection_ids_issued_client,migration_type,path_validation_initiated,connection_close_type,bytes_sent_client,bytes_sent_server,packets_sent_client,packets_sent_server,quic_packets_sent_client,quic_packets_sent_server,handshake_duration,time_to_migration,migration_duration,packets_before_migration,app_data_bytes_before_migration,padding_bytes_in_validation_pc,padding_bytes_in_validation_pr,mtu,total_bidi_streams_client_init,total_bidi_streams_server_init,total_udi_streams_client_init,total_udi_streams_server_init,bytes_bidi_streams_client_init_client_sent,bytes_bidi_streams_client_init_server_sent,bytes_bidi_streams_server_init_client_sent,bytes_bidi_streams_server_init_server_sent,bytes_udi_streams_client_init,bytes_udi_streams_server_init,ack_sent_client,ack_sent_server,crypto_sent_client,crypto_sent_server,handshake_done_client,handshake_done_server,path_challenge_sent_client,path_challenge_sent_server,path_response_sent_client,path_response_sent_server
0,1,127.0.0.2,127.0.0.1,47264,4433,2025-10-12T15-46-56.855591,2025-10-12T15-46-56.865064,9.472847,1,1,1,0,0,1,IP_AND_PORT,1,CLIENT_CLOSE,7402,4597,13,11,18,15,6.505966,7.506847,0.629187,13,47,1303,0,1398,1,0,4,4,72,115,0,0,47,47,4,4,3,4,0,1,1,0,1,0


#### Load all jsons from a directory

In [9]:
directory_path = r'captures_json/quiche'
all_extracted_features = []

In [10]:
json_files = [f for f in os.listdir(directory_path) if f.endswith('.json')]

In [11]:
for filename in json_files:
    full_path = os.path.join(directory_path, filename)
    print(f"Processing '{filename}'...")
    
    # Process the file and get the features dictionary
    features = process_quic_capture(full_path)
    
    # If features were successfully extracted, add them to our list
    if features:
        all_extracted_features.append(features)
        print(f"-> Successfully extracted features from '{filename}'.\n")
    


Processing 'quiche_capture_1.json'...
-> Successfully extracted features from 'quiche_capture_1.json'.

Processing 'quiche_capture_10.json'...
-> Successfully extracted features from 'quiche_capture_10.json'.

Processing 'quiche_capture_11.json'...
-> Successfully extracted features from 'quiche_capture_11.json'.

Processing 'quiche_capture_12.json'...
-> Successfully extracted features from 'quiche_capture_12.json'.

Processing 'quiche_capture_13.json'...
-> Successfully extracted features from 'quiche_capture_13.json'.

Processing 'quiche_capture_14.json'...
-> Successfully extracted features from 'quiche_capture_14.json'.

Processing 'quiche_capture_15.json'...
-> Successfully extracted features from 'quiche_capture_15.json'.

Processing 'quiche_capture_16.json'...
-> Successfully extracted features from 'quiche_capture_16.json'.

Processing 'quiche_capture_17.json'...
-> Successfully extracted features from 'quiche_capture_17.json'.

Processing 'quiche_capture_18.json'...
-> Succes

In [12]:
print("------------------------------------------")
print("Finished processing all files.")

if all_extracted_features:
    df = pd.DataFrame(all_extracted_features)
    
    # Optional: Set one of the columns as the index
    # df.set_index('ID', inplace=True)

    print("Final DataFrame with all extracted features:")
    display(df) # Use display() if in a Jupyter Notebook, otherwise use print(df)
else:
    print("No features were extracted. The final DataFrame is empty.")

------------------------------------------
Finished processing all files.
Final DataFrame with all extracted features:


,file_id,initial_ip_client,initial_ip_server,initial_port_client,initial_port_server,time_first,time_last,connection_duration,version_negotiation_occurred,retry_occurred,new_connection_ids_issued_server,retired_cid_count_client,retired_cid_count_server,new_connection_ids_issued_client,migration_type,path_validation_initiated,first_path_validation_response_latency,connection_close_type,bytes_sent_client,bytes_sent_server,packets_sent_client,packets_sent_server,quic_packets_sent_client,quic_packets_sent_server,handshake_duration,time_to_migration,migration_duration,packets_before_migration,app_data_bytes_before_migration,padding_bytes_in_validation_pc,padding_bytes_in_validation_pr,mtu,total_bidi_streams_client_init,total_bidi_streams_server_init,total_udi_streams_client_init,total_udi_streams_server_init,bytes_bidi_streams_client_init_client_sent,bytes_bidi_streams_client_init_server_sent,bytes_bidi_streams_server_init_client_sent,bytes_bidi_streams_server_init_server_sent,bytes_udi_streams_client_init,bytes_udi_streams_server_init,ack_sent_client,ack_sent_server,crypto_sent_client,crypto_sent_server,handshake_done_client,handshake_done_server,path_challenge_sent_client,path_challenge_sent_server,path_response_sent_client,path_response_sent_server
0,1,127.0.0.2,127.0.0.1,54135,4433,2025-10-20T18-31-41.865828,2025-10-20T18-31-41.973080,107.251883,1,1,1,0,0,1,IP_AND_PORT,1,1.174212,CLIENT_CLOSE,7269,4649,14,12,19,19,77.664852,86.835861,2.040148,13,47,1294,1294,1382,1,0,4,4,72,308,0,0,47,47,5,5,3,4,0,1,1,1,1,1
1,10,127.0.0.2,127.0.0.1,51029,4433,2025-10-20T18-32-15.423191,2025-10-20T18-32-15.437308,14.117002,1,1,1,0,0,1,IP_AND_PORT,1,0.800133,CLIENT_CLOSE,7269,4648,14,12,19,19,8.035898,10.459900,1.227140,13,47,1294,1294,1382,1,0,4,4,72,308,0,0,47,47,5,5,3,4,0,1,1,1,1,1
2,11,127.0.0.2,127.0.0.1,53788,4433,2025-10-20T18-32-18.998389,2025-10-20T18-32-19.023298,24.909019,1,1,1,0,0,1,IP_AND_PORT,1,1.365900,CLIENT_CLOSE,7269,4649,14,12,19,19,15.118837,18.115997,2.296925,13,47,1294,1294,1382,1,0,4,4,72,308,0,0,47,47,5,5,3,4,0,1,1,1,1,1
3,12,127.0.0.2,127.0.0.1,64736,4433,2025-10-20T18-32-23.160335,2025-10-20T18-32-23.177023,16.687870,1,1,1,0,0,1,IP_AND_PORT,1,0.665188,CLIENT_CLOSE,7269,4648,14,12,19,19,11.420012,13.275862,1.053095,13,47,1294,1294,1382,1,0,4,4,72,308,0,0,47,47,5,5,3,4,0,1,1,1,1,1
4,13,127.0.0.2,127.0.0.1,63284,4433,2025-10-20T18-32-26.815987,2025-10-20T18-32-26.834053,18.065929,1,1,1,0,0,1,IP_AND_PORT,1,0.712156,CLIENT_CLOSE,7269,4649,14,12,19,19,10.921955,12.638807,1.119137,12,21,1294,1294,1382,1,0,4,4,72,308,0,0,47,47,5,5,3,4,0,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,87,127.0.0.2,127.0.0.1,52460,4433,2025-10-15T11-48-44.321248,2025-10-15T11-48-44.329357,8.108854,1,1,1,0,0,1,IP_AND_PORT,1,0.263929,CLIENT_CLOSE,7269,4411,14,12,19,19,5.815029,6.562948,0.427008,13,47,1294,1294,1382,1,0,4,4,72,73,0,0,47,47,5,5,3,4,0,1,1,1,1,1
86,88,127.0.0.2,127.0.0.1,63564,4433,2025-10-15T11-49-51.290642,2025-10-15T11-49-51.296610,5.968094,1,1,1,0,0,1,IP_AND_PORT,1,0.204086,CLIENT_CLOSE,7269,4411,14,12,19,19,4.128218,4.734993,0.340939,13,47,1294,1294,1382,1,0,4,4,72,73,0,0,47,47,5,5,3,4,0,1,1,1,1,1
87,89,127.0.0.2,127.0.0.1,53698,4433,2025-10-15T11-51-09.800296,2025-10-15T11-51-09.808605,8.308887,1,1,1,0,0,1,IP_AND_PORT,1,0.275135,CLIENT_CLOSE,7194,4411,13,12,18,19,5.638123,6.518841,0.509977,13,47,1294,1294,1382,1,0,4,4,72,73,0,0,47,47,4,5,3,4,0,1,1,1,1,1
88,9,127.0.0.2,127.0.0.1,50845,4433,2025-10-20T18-32-11.709787,2025-10-20T18-32-11.718859,9.072065,1,1,1,0,0,1,IP_AND_PORT,1,0.509024,CLIENT_CLOSE,7269,4648,14,12,19,19,4.799843,6.312132,0.849009,13,47,1294,1294,1382,1,0,4,4,72,308,0,0,47,47,5,5,3,4,0,1,1,1,1,1


In [14]:
df.to_csv('high_level_features/quiche/quiche_all_extracted_high_level_features.csv', index=False)